# Simple BERTopic Pipeline (Hippocorpus)

1. Configure topic mode and load data.
2. Optional BERTopic-native LLM representation for topic names during modeling.
3. Fit BERTopic and export topic outputs.
4. Optional post-fit LLM suggestion and review workflow for topic names.
5. Label the full Hippocorpus with regex rules.
6. Append labels to reduced datasets by statement index.

In [ ]:
from pathlib import Path
import os
import re
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

## Step 1: Configure topic fitting mode and load data
- Set `TOPIC_MODE` (`'hdbscan'` or `'kmeans'`).
- Configure paths and `APPEND_TARGETS`.
- Load Hippocorpus into `hip` and `docs`.

In [ ]:
DATA_DIR = Path('../data')
HIPPOCORPUS_PATH = DATA_DIR / 'hippocorpus.csv'

# Topic fitting mode: 'hdbscan' (default) or 'kmeans'
TOPIC_MODE = 'hdbscan'
KMEANS_N_CLUSTERS = 30

OUTPUT_DOCS_PATH = DATA_DIR / ('output_k_means.csv' if TOPIC_MODE == 'kmeans' else 'output.csv')
OUTPUT_TOPICS_PATH = DATA_DIR / ('topic_info_k_means.csv' if TOPIC_MODE == 'kmeans' else 'topic_info.csv')
LABELED_HIPPOCORPUS_PATH = DATA_DIR / 'hippocorpus_labeled.csv'

# Add or remove targets here whenever you want to append labels to new files.
APPEND_TARGETS = [
    {'input': 'final.csv', 'index_col': 'os_id', 'output': 'final_labeled.csv'},
    {'input': 'final_deltamax.csv', 'index_col': 'os_id', 'output': 'final_deltamax_labeled.csv'},
]

hip = pd.read_csv(HIPPOCORPUS_PATH, sep=',')
docs = hip['text'].astype(str).tolist()
print(f'Hippocorpus rows: {len(hip):,}')

## Step 2 (Optional): Use BERTopic LLM representation model

This is the BERTopic-native approach to naming topics during modeling.

- Set `USE_BERTOPIC_LLM_REPRESENTATION = True`.
- Export `OPENAI_API_KEY` before running.
- If disabled or unavailable, the pipeline falls back to `KeyBERTInspired`.

In [ ]:
USE_BERTOPIC_LLM_REPRESENTATION = False
BERTOPIC_LLM_MODEL = 'gpt-4.1-mini'

REPRESENTATION_MODEL = KeyBERTInspired()

if USE_BERTOPIC_LLM_REPRESENTATION:
    try:
        from openai import OpenAI
        from bertopic.representation import OpenAI as BERTopicOpenAI
    except ImportError as exc:
        raise ImportError(
            'Install openai to use BERTopic OpenAI representation: pip install openai'
        ) from exc

    api_key = os.getenv('OPENAI_API_KEY', '')
    if not api_key:
        raise ValueError('Missing OPENAI_API_KEY environment variable.')

    client = OpenAI(api_key=api_key)
    prompt = (
        'You are an expert topic modeler. Given topic keywords, generate one concise '\
        'human-readable topic label (2-3 words). Return only the label text.'
    )
    REPRESENTATION_MODEL = BERTopicOpenAI(
        client,
        model=BERTOPIC_LLM_MODEL,
        prompt=prompt,
        chat=True,
    )
    print(f'Using BERTopic OpenAI representation model: {BERTOPIC_LLM_MODEL}')
else:
    print('Using default BERTopic representation model: KeyBERTInspired')

## Step 3: Fit topics

In [ ]:
representation_model = REPRESENTATION_MODEL if 'REPRESENTATION_MODEL' in globals() else KeyBERTInspired()

if TOPIC_MODE == 'kmeans':
    cluster_model = KMeans(n_clusters=KMEANS_N_CLUSTERS, random_state=42)
    vectorizer_model = CountVectorizer(stop_words='english')

    topic_model = BERTopic(
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
    )
    topics, probs = topic_model.fit_transform(docs)

    # Old kmeans pipeline: no outlier reduction step
    hip['topic'] = topics
else:
    topic_model = BERTopic(representation_model=representation_model)
    topics, probs = topic_model.fit_transform(docs)

    # Old hdbscan pipeline: reduce outliers then update topic representations
    reduced_topics = topic_model.reduce_outliers(docs, topics)
    hip['topic'] = reduced_topics
    topic_model.update_topics(docs, topics=reduced_topics)

hip.to_csv(OUTPUT_DOCS_PATH, index=False)
topic_info = topic_model.get_topic_info()
topic_info.to_csv(OUTPUT_TOPICS_PATH, index=False)

print(f'Saved: {OUTPUT_DOCS_PATH}')
print(f'Saved: {OUTPUT_TOPICS_PATH}')
topic_info.head(10)

## Step 4 (Optional Alternative): Post-fit LLM topic name suggestions

Use this step only if you want a separate review/apply loop after fitting topics.

- Set `USE_LLM_TOPIC_NAMES = True`.
- Export your key before running: `export OPENAI_API_KEY=...`
- Review `data/topic_name_suggestions.csv` before applying any labels.

The next code cell only creates suggestions; it does not change your dataset labels automatically.

In [ ]:
import json
import os
from urllib import error, request

# Set to True to request suggested topic names from an OpenAI-compatible API.
USE_LLM_TOPIC_NAMES = False
LLM_API_KEY = os.getenv('OPENAI_API_KEY', '')
LLM_BASE_URL = 'https://api.openai.com/v1'
LLM_MODEL = 'gpt-4.1-mini'
LLM_OUTPUT_PATH = DATA_DIR / 'topic_name_suggestions.csv'


def build_topic_payload(topic_info_df, max_topics=40):
    keep = topic_info_df[topic_info_df['Topic'] != -1].head(max_topics).copy()
    payload = []
    for row in keep.itertuples(index=False):
        representation = row.Representation
        if not isinstance(representation, list):
            representation = [tok.strip() for tok in str(representation).strip('[]').replace("'", '').split(',') if tok.strip()]
        payload.append(
            {
                'topic_id': int(row.Topic),
                'count': int(row.Count),
                'current_name': str(row.Name),
                'keywords': representation[:10],
            }
        )
    return payload


if not USE_LLM_TOPIC_NAMES:
    print('LLM naming disabled. Set USE_LLM_TOPIC_NAMES = True to generate suggestions.')
else:
    if not LLM_API_KEY:
        raise ValueError('Missing OPENAI_API_KEY environment variable.')

    topics_payload = build_topic_payload(topic_info)
    prompt = {
        'task': 'Suggest short, human-readable labels for BERTopic topics.',
        'rules': [
            'Label length: 2-5 words.',
            'Be specific and concrete.',
            'Avoid numbering in the label text.',
            'Return valid JSON only.',
        ],
        'return_format': {
            'topic_names': {
                '<topic_id>': '<suggested_label>'
            }
        },
        'topics': topics_payload,
    }

    body = {
        'model': LLM_MODEL,
        'temperature': 0.2,
        'messages': [
            {'role': 'system', 'content': 'You rename topic-model clusters for analytics reporting.'},
            {'role': 'user', 'content': json.dumps(prompt)},
        ],
    }

    req = request.Request(
        f"{LLM_BASE_URL.rstrip('/')}/chat/completions",
        data=json.dumps(body).encode('utf-8'),
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {LLM_API_KEY}',
        },
        method='POST',
    )

    try:
        with request.urlopen(req, timeout=90) as resp:
            raw = resp.read().decode('utf-8')
    except error.HTTPError as exc:
        detail = exc.read().decode('utf-8', errors='ignore')
        raise RuntimeError(f'LLM API error {exc.code}: {detail}') from exc

    response_json = json.loads(raw)
    content = response_json['choices'][0]['message']['content']

    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        start = content.find('{')
        end = content.rfind('}')
        if start == -1 or end == -1:
            raise ValueError('Model response was not valid JSON.')
        parsed = json.loads(content[start : end + 1])

    mapping = parsed.get('topic_names', parsed)

    suggestions = []
    for topic_id, label in mapping.items():
        try:
            topic_int = int(topic_id)
        except (TypeError, ValueError):
            continue
        suggestions.append({'Topic': topic_int, 'suggested_label': str(label).strip()})

    suggestion_df = pd.DataFrame(suggestions).drop_duplicates(subset='Topic')
    suggestion_df = topic_info[['Topic', 'Name', 'Count']].merge(suggestion_df, on='Topic', how='left')
    suggestion_df.to_csv(LLM_OUTPUT_PATH, index=False)

    print(f'Saved: {LLM_OUTPUT_PATH}')
    suggestion_df.head(20)

In [ ]:
# Optional: apply reviewed topic names back to topic_info.
# Set APPLY_LLM_NAMES = True only after checking LLM_OUTPUT_PATH.
APPLY_LLM_NAMES = False
FINAL_TOPIC_INFO_PATH = DATA_DIR / ('topic_info_k_means_named.csv' if TOPIC_MODE == 'kmeans' else 'topic_info_named.csv')

if not APPLY_LLM_NAMES:
    print('Apply step disabled. Set APPLY_LLM_NAMES = True to write named topic info.')
else:
    if not LLM_OUTPUT_PATH.exists():
        raise FileNotFoundError(f'Missing suggestion file: {LLM_OUTPUT_PATH}')

    reviewed = pd.read_csv(LLM_OUTPUT_PATH)
    required_cols = {'Topic', 'suggested_label'}
    missing_cols = required_cols.difference(reviewed.columns)
    if missing_cols:
        raise ValueError(f'Suggestion file is missing columns: {sorted(missing_cols)}')

    reviewed = reviewed[['Topic', 'suggested_label']].copy()
    reviewed['Topic'] = pd.to_numeric(reviewed['Topic'], errors='coerce').astype('Int64')
    reviewed['suggested_label'] = reviewed['suggested_label'].astype(str).str.strip()
    reviewed = reviewed.dropna(subset=['Topic'])
    reviewed = reviewed[reviewed['suggested_label'].ne('')]
    reviewed = reviewed[reviewed['suggested_label'].str.lower().ne('nan')]
    reviewed = reviewed.drop_duplicates(subset='Topic', keep='last')

    topic_info_named = topic_info.copy()
    topic_info_named = topic_info_named.merge(
        reviewed.rename(columns={'suggested_label': 'llm_label'}),
        on='Topic',
        how='left',
    )
    topic_info_named['final_name'] = topic_info_named['llm_label'].fillna(topic_info_named['Name'])

    topic_info_named.to_csv(FINAL_TOPIC_INFO_PATH, index=False)
    print(f'Saved: {FINAL_TOPIC_INFO_PATH}')
    topic_info_named[['Topic', 'Name', 'llm_label', 'final_name', 'Count']].head(20)

## Step 5: Create regex rules from inspected topics and label full Hippocorpus
Edit only `RULES` below. First match wins.

In [ ]:
RULES = [
    (1, 'Funeral/Death', re.compile(
        r'\b(funeral|wake|burial|cremat(?:e|ion|ed)|cemeter(?:y|ies)|died|death|passed away|grief|mourning|eulog(?:y|ies)|memorial)\b',
        re.I,
    )),
    (2, 'Hospital/Injury', re.compile(
        r'\b(hospital|doctor|dr\.?|nurse|clinic|er|emergency(?: room)?|ambulance|surgery|operation|radiation|chemo(?:therapy)?|cancer|tumou?r|treatment|therapy|medication|prescription|diagnos(?:is|ed)|pain|injur(?:y|ed)|hurt|wound|bleed(?:ing)?|stitch(?:es)?|broken|fracture|sprain|concussion|infection|fever)\b',
        re.I,
    )),
    (3, 'Money/Gambling', re.compile(
        r'\b(casino|gambl(?:e|ing)|slot(?:s)?|bet(?:s|ting)?|jackpot|poker|blackjack|roulette|lottery|winnings?|won (?:big|money)|debt|loan|overdraft|credit card|bank account|bills?|payment(?:s)?)\b',
        re.I,
    )),
    (4, 'NaturalDisaster/Weather', re.compile(
        r'\b(hurricane|tornado|earthquake|aftershock|wildfire|evacuat(?:e|ed|ion)|flood(?:ed|ing)?|storm surge|blizzard|ice storm|hailstorm|water damage|house flooded)\b',
        re.I,
    )),
    (5, 'Children', re.compile(
        r'\b(baby|newborn|toddler|child|children|kid|kids|pregnan(?:t|cy)|expecting|labor|delivery|gave birth|born|birth|ultrasound|due date|midwife|diaper(?:s)?|daycare|crib|stroller)\b',
        re.I,
    )),
    (6, 'Pets', re.compile(
        r'\b(dog|puppy|cat|kitten|pet|pets|vet|veterinarian|animal hospital|leash|litter|adopt(?:ed|ion)?|groom(?:ed|er|ing)?|walk(?:ed|ing)? (?:the )?dog)\b',
        re.I,
    )),
    (7, 'Vacation/Trip', re.compile(
        r'\b(vacation|holiday|trip|road trip|travel(?:ed|ing)?|flight|airport|hotel|resort|airbnb|beach|mountain(?:s)?|hike(?:d|ing)?|trail|camp(?:ing)?|cabin|lake|river|kayak(?:ing)?|raft(?:ing)?|fishing|bait|national park|ski(?:ing)?|snowboard(?:ing)?|surf(?:ing)?|paris|london|europe|new york)\b',
        re.I,
    )),
    (8, 'Work', re.compile(
        r'\b(job|work|company|boss|manager|coworker(?:s)?|colleague(?:s)?|office|shift|meeting(?:s)?|deadline|project|client|promotion|promoted|raise|salary|position|appl(?:y|ied|ication)|interview(?:s)?|resume|cv|hired|fired|laid off|quit|resign(?:ed|ation)?)\b',
        re.I,
    )),
    (9, 'Birthday', re.compile(
        r'\b(birthday|bday|surprise party|party|cake|candles|balloons|present(?:s)?|gift(?:s)?|blew out (?:the )?candles)\b',
        re.I,
    )),
    (10, 'Sports', re.compile(
        r'\b(playoffs|tournament|match|season|team|coach|practice|race|marathon|triathlon|training|workout|gym|baseball|basketball|soccer|football|tennis|hockey|swim(?:ming)?|cycling|running|lift(?:ing)?|weights?)\b',
        re.I,
    )),
    (11, 'Concert/Music', re.compile(
        r'\b(concert|gig|live music|show|band|music festival|festival|setlist|encore|venue|tickets?)\b',
        re.I,
    )),
    (12, 'School', re.compile(
        r'\b(university|college|school|campus|class(?:es)?|lecture(?:s)?|teacher|professor|homework|assignment(?:s)?|exam(?:s)?|test(?:s)?|final(?:s)?|graduat(?:e|ed|ion)|degree|diploma|ceremony)\b',
        re.I,
    )),
    (13, 'Home/Moving', re.compile(
        r'\b(house|home|apartment|flat|condo|roommate(?:s)?|mortgage|lease|rent|landlord|tenant|moved in|move(?:d|s|ing)?|moving|packing|boxes|unpack(?:ed|ing)?|new place|new house|new apartment)\b',
        re.I,
    )),
    (14, 'Relationships', re.compile(
        r'\b(wife|husband|partner|fianc(?:e|ée)|boyfriend|girlfriend|dating|date night|single|break(?:\s?up)?|broke up|marry|married|wedding|engag(?:e|ed|ement)|proposal|propos(?:e|al)|ring|in love|love)\b',
        re.I,
    )),
    (15, 'Family', re.compile(
        r'\b(mom|mother|mum|dad|father|parents?|stepmom|stepmother|stepdad|stepfather|sister|brother|siblings?|daughter|son|grandma|grandpa|grandmother|grandfather|aunt|uncle|cousin(?:s)?|niece|nephew|in-?laws?|family reunion|custody)\b',
        re.I,
    )),
    (16, 'Writing', re.compile(
        r'\b(diary|journal|journaling|dear diary|book(?:s)?|novel|chapter|story|writ(?:e|ing|ten)|author|manuscript|publish(?:ed|ing)?|poem(?:s)?|poetry|library)\b',
        re.I,
    )),
]

def label_text(text):
    if pd.isna(text):
        return 0, 'Other'
    for topic_id, topic_label, pattern in RULES:
        if pattern.search(str(text)):
            return topic_id, topic_label
    return 0, 'Other'

In [ ]:
hip = pd.read_csv(HIPPOCORPUS_PATH)
hip[['topic_id', 'topic_label']] = hip['text'].apply(lambda x: pd.Series(label_text(x)))
hip.to_csv(LABELED_HIPPOCORPUS_PATH, index=False)

print(f'Saved: {LABELED_HIPPOCORPUS_PATH}')
hip['topic_label'].value_counts(dropna=False).head(20)

## Step 6: Append labels to reduced datasets by statement index

In [ ]:
hip_labeled = pd.read_csv(LABELED_HIPPOCORPUS_PATH)

# Statement index alignment in the labeled Hippocorpus
hip_labeled['index'] = hip_labeled['index'].astype(str)
labels = hip_labeled[['index', 'topic_id', 'topic_label']].drop_duplicates(subset='index')

for target in APPEND_TARGETS:
    input_path = DATA_DIR / target['input']
    output_path = DATA_DIR / target['output']
    index_col = target['index_col']

    reduced = pd.read_csv(input_path)
    reduced[index_col] = reduced[index_col].astype(str)

    merged = reduced.merge(labels, left_on=index_col, right_on='index', how='left').drop(columns=['index'])
    missing = merged['topic_label'].isna().sum()

    merged.to_csv(output_path, index=False)
    print(f'Saved: {output_path} | Missing labels: {missing}')

Outputs:
- data/output.csv (or data/output_k_means.csv)
- data/topic_info.csv (or data/topic_info_k_means.csv)
- data/topic_name_suggestions.csv (optional LLM step)
- data/topic_info_named.csv (or data/topic_info_k_means_named.csv, optional apply step)
- data/hippocorpus_labeled.csv
- one labeled output per APPEND_TARGETS entry